# 07 Ablation and Controlled Prior-Drift Analysis

Runs matched component ablations on the real stage-05 protocol and evaluates stage-05/stage-06 models under controlled held-out label-prior shifts. The drift study is explicitly a prior-shift robustness test, not a claim of natural temporal drift or continual-learning forgetting.

In [1]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

ABS_PROJECT_ROOT = Path('/Volumes/LocalDrive1/onlyscholar-projects/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
EXPECTED_DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION')
DRIVE_PROJECT_ROOT = EXPECTED_DRIVE_PROJECT_ROOT

def is_project_root(path):
    return (path / 'configs' / 'project_colab.json').is_file() and (path / 'src' / 'ecotrust').is_dir()

def discover_mounted_project():
    candidates = [EXPECTED_DRIVE_PROJECT_ROOT]
    my_drive = Path('/content/drive/MyDrive')
    candidates.extend([my_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION', my_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION'])
    shared_drives = Path('/content/drive/Shareddrives')
    if shared_drives.is_dir():
        for shared_drive in shared_drives.iterdir():
            candidates.extend([shared_drive / 'projects' / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION', shared_drive / 'implementation' / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION', shared_drive / 'EcoTrust-FLIDS[DeepakSir]' / 'IMPLEMENTATION'])
    return next((candidate for candidate in candidates if is_project_root(candidate)), None)

try:
    from google.colab import drive  # type: ignore
    if not is_project_root(EXPECTED_DRIVE_PROJECT_ROOT):
        print('Drive project is not visible; resetting stale DriveFS state...', flush=True)
        try:
            drive.flush_and_unmount()
        except Exception:
            pass
        subprocess.run(['fusermount', '-uz', '/content/drive'], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        shutil.rmtree('/content/drive', ignore_errors=True)
        Path('/content/drive').mkdir(parents=True, exist_ok=True)
        drive.mount('/content/drive', force_remount=False, timeout_ms=120000)
    discovered_root = discover_mounted_project()
    if discovered_root is not None:
        DRIVE_PROJECT_ROOT = discovered_root
except ImportError:
    pass
except Exception as exc:
    raise RuntimeError(f'Google Drive project mount failed: {type(exc).__name__}: {exc}') from exc

def find_project_root():
    candidates = []
    env_root = os.environ.get('ECOTRUST_PROJECT_ROOT')
    if env_root:
        candidates.append(Path(env_root))
    candidates.extend([Path.cwd(), Path.cwd().parent, DRIVE_PROJECT_ROOT, ABS_PROJECT_ROOT])
    for base in list(candidates):
        candidates.extend(base.parents)
    seen = set()
    for candidate in candidates:
        expanded = candidate.expanduser()
        resolved = expanded.resolve() if expanded.exists() else expanded
        if str(resolved) in seen:
            continue
        seen.add(str(resolved))
        if (resolved / 'configs' / 'project.json').exists() and (resolved / 'src' / 'ecotrust').exists():
            return resolved
    raise RuntimeError('Cannot find the EcoTrust-FLIDS IMPLEMENTATION folder in the mounted Google Drive.')

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
CONFIG_PATH = PROJECT_ROOT / 'configs' / ('project_colab.json' if str(PROJECT_ROOT).startswith('/content/drive') else 'project.json')
RUN_MODE = 'full'
DEVICE = 'cpu'
RESUME = True
DATASETS = ['Edge-IIoTset', 'TON_IoT', 'CICIoT2023']
BUDGET_FRACTION = 0.6

importlib.invalidate_caches()
import ecotrust.agent.controller as controller_module
import ecotrust.evaluation.ablation_drift as ablation_module
importlib.reload(controller_module)
importlib.reload(ablation_module)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CONFIG_PATH:', CONFIG_PATH)
print('RUN_MODE:', RUN_MODE)
print('DEVICE:', DEVICE)
print('RESUME:', RESUME)
print('ABLATION_SOURCE:', ablation_module.__file__)

result = ablation_module.run_with_failure_manifest(CONFIG_PATH, run_mode=RUN_MODE, datasets=DATASETS, device=DEVICE, budget_fraction=BUDGET_FRACTION, resume=RESUME)
print('STAGE_07_STATUS:', result['status'])
print('STAGE_07_MANIFEST:', result['manifest'])


PROJECT_ROOT: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION
CONFIG_PATH: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/configs/project_colab.json
RUN_MODE: full
DEVICE: cpu
RESUME: True
ABLATION_SOURCE: /content/drive/MyDrive/projects/implementation/EcoTrust-FLIDS[DeepakSir]/IMPLEMENTATION/src/ecotrust/evaluation/ablation_drift.py
Ablation Edge-IIoTset no_trust
[2026-10-23T08:21:10+00:00] Starting 30 federated rounds on cpu; model_size_mb=0.1426.
[2026-10-23T08:21:15+00:00] Round 1/30 complete: f1=1.000000, selected=12/20, energy_saving=49.61%.
[2026-10-23T08:21:18+00:00] Round 2/30 complete: f1=1.000000, selected=12/20, energy_saving=49.61%.
[2026-10-23T08:21:22+00:00] Round 3/30 complete: f1=1.000000, selected=12/20, energy_saving=49.61%.
[2026-10-23T08:21:25+00:00] Round 4/30 complete: f1=1.000000, selected=12/20, energy_saving=49.61%.
[2026-10-23T08:21:28+00:00] Round 5/30 complete: f1=1.000000, sel